### Load the data

In [4]:
import pandas as pd
import os

# Load the race results
# Adjust path based on actual file structure
race_results_path = '/Users/aylinyasgul/IE/NASCAR/nascar-visibility-analysis/data/external/nascar-dataset/NASCAR 2017-2024 Full Race  Points Data - Cup.csv'

# Check if file exists
if os.path.exists(race_results_path):
    df = pd.read_csv(race_results_path)
    print(f"Loaded {len(df)} rows of data")
else:
    print(f"File not found: {race_results_path}")
    print("Available files in data/external/:")
    for root, dirs, files in os.walk('data/external/'):
        for file in files:
            print(f"  {os.path.join(root, file)}")

Loaded 11325 rows of data


### Examine data structure

In [5]:
# Basic exploration
print("=== Dataset Overview ===")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Preview
print("\n=== First 5 Rows ===")
print(df.head())

# Data types
print("\n=== Data Types ===")
print(df.dtypes)

# Check for missing values
print("\n=== Missing Values ===")
print(df.isnull().sum())

=== Dataset Overview ===
Shape: (11325, 18)
Columns: ['year', 'race_num', 'track', 'track_type', 'fin', 'start', 'car_num', 'driver', 'manu', 'team_name', 'laps', 'laps_led', 'status', 'points', 'stage_1', 'stage_2', 'stage_3_or_duel', 'stage_points']

=== First 5 Rows ===
   year  race_num    track     track_type  fin  start  car_num  \
0  2017        -1  Daytona  superspeedway    1      1       24   
1  2017        -1  Daytona  superspeedway    2     12        1   
2  2017        -1  Daytona  superspeedway    3      4        4   
3  2017        -1  Daytona  superspeedway    4      2        2   
4  2017        -1  Daytona  superspeedway    5      5       20   

            driver       manu             team_name  laps  laps_led   status  \
0    Chase Elliott  Chevrolet  Hendrick Motorsports    60        25  running   
1   Jamie McMurray  Chevrolet   Chip Ganassi Racing    60         0  running   
2    Kevin Harvick       Ford   Stewart-Haas Racing    60         0  running   
3  Brad K

### Filter for 2024 race season

In [8]:
# Identify the year/season column
# Common column names: 'year', 'Year', 'Season', 'Date' (extract year)

# If there's a year column (lowercase):
if 'year' in df.columns:
    df_2024 = df[df['year'] == 2024]
# If there's a Year column (capitalized):
elif 'Year' in df.columns:
    df_2024 = df[df['Year'] == 2024]
# If there's a Date column:
elif 'Date' in df.columns:
    df['year'] = pd.to_datetime(df['Date']).dt.year
    df_2024 = df[df['year'] == 2024]
else:
    print("Need to identify how to filter to 2024")
    print("Available columns:", df.columns.tolist())
    df_2024 = df.copy()  # fallback: use all data

print(f"\n2024 Season: {len(df_2024)} rows")

# Verify we have all races
track_col = 'track' if 'track' in df_2024.columns else 'Race'
races_2024 = df_2024[track_col].unique()
print(f"Unique races in 2024: {len(races_2024)}")
print("\nRace list:")
for race in sorted(races_2024):
    print(f"  - {race}")


2024 Season: 1398 rows
Unique races in 2024: 26

Race list:
  - Atlanta
  - Bristol
  - COTA
  - Charlotte
  - Charlotte Roval
  - Chicago Street Course
  - Darlington
  - Daytona
  - Dover
  - Gateway
  - Homestead
  - Indianapolis
  - Iowa
  - Kansas
  - Las Vegas
  - Martinsville
  - Michigan
  - Nashville
  - New Hampshire
  - Phoenix
  - Pocono
  - Richmond
  - Sonoma
  - Talladega
  - Texas
  - Watkins Glen


In [11]:
# 2024 NASCAR Cup Series should have 36 races
expected_races = 36

actual_races = df_2024['race_num'].nunique()  # Adjust column name

print(f"Expected races: {expected_races}")
print(f"Actual races: {actual_races}")

if actual_races >= expected_races:
    print("Dataset appears complete for 2024 season")
else:
    print(f"WARNING: Missing {expected_races - actual_races} races")
    print("May need to supplement with Racing Reference scraping")

Expected races: 36
Actual races: 38
Dataset appears complete for 2024 season


In [12]:
# Save 2024 data to raw folder
output_path = '../data/raw/race_results_2024_kaggle.csv'
df_2024.to_csv(output_path, index=False)
print(f"Saved {len(df_2024)} rows to {output_path}")

Saved 1398 rows to ../data/raw/race_results_2024_kaggle.csv


In [15]:
# Load your data (adjust path as needed)
# df = pd.read_csv('data/raw/race_results_2024_kaggle.csv')
# OR
# df = pd.read_csv('data/raw/race_results_2024_scraped.csv')

print("=== Data Quality Report ===\n")

# 1. Check race count
race_count = df['race_num'].nunique()
print(f"Unique races: {race_count} (expected: 36)")

# 2. Check for missing values
print(f"\nMissing values:")
for col in df.columns:
    missing = df[col].isnull().sum()
    if missing > 0:
        print(f"  {col}: {missing} ({missing/len(df)*100:.1f}%)")

# 3. Check driver count per race
drivers_per_race = df.groupby('race_num').size()
print(f"\nDrivers per race:")
print(f"  Min: {drivers_per_race.min()}")
print(f"  Max: {drivers_per_race.max()}")
print(f"  Average: {drivers_per_race.mean():.1f}")

# 4. Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

# 5. Sample data check
print("\n=== Sample Data ===")
print(df.sample(5))

=== Data Quality Report ===

Unique races: 38 (expected: 36)

Missing values:
  points: 337 (3.0%)
  stage_1: 2812 (24.8%)
  stage_2: 2812 (24.8%)
  stage_3_or_duel: 10913 (96.4%)

Drivers per race:
  Min: 168
  Max: 320
  Average: 298.0

Duplicate rows: 0

=== Sample Data ===
      year  race_num         track         track_type  fin  start  car_num  \
101   2017         2       Atlanta       intermediate   20     37       72   
9352  2023        21        Pocono  long intermediate   12     27       10   
8504  2022        35  Martinsville        short track   12      7       99   
3489  2019        15      Michigan  long intermediate   21     29       95   
5074  2020        19        Kansas       intermediate    7     24       41   

                driver       manu              team_name  laps  laps_led  \
101         Cole Whitt  Chevrolet    TriStar Motorsports   324         0   
9352     Aric Almirola       Ford    Stewart-Haas Racing   160         0   
8504     Daniel Suarez  C